# 🔍 Deepfake Detection — Optimized Pipeline
### EfficientNet-B4 · AMP · Val-based Callbacks · Checkpoint Resume · TTA Inference

**Datasets**: FaceForensics++, CelebDF-V2, DFD, Wild-Deepfake  
**Key Upgrades**:
- `EfficientNet-B4` (vs B0) — much better feature extraction
- Mixed-precision training (`torch.amp`) — faster + more VRAM-efficient
- Proper train/val split — val F1 drives model selection (no train leakage)
- Full checkpoint saving (model + optimizer + scheduler + scaler)
- **Resume training**: loads `best_model.pth` on startup and continues
- Face-crop option via `facenet-pytorch` MTCNN
- JPEG-noise & RandomErasing augmentations
- Test-Time Augmentation (TTA) for inference
- Gradient clipping, `cudnn.benchmark`, `non_blocking` transfers


In [1]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\Neel\\AppData\\Local\\Temp\\pip-unpack-inyo5bas\\torch-2.10.0+cu126-cp310-cp310-win_amd64.whl'
Consider using the `--user` option or check the permissions.



In [8]:
# ═══════════════════════════════════════════════════════
#  FRAME EXTRACTION  —  skip this cell if already done
# ═══════════════════════════════════════════════════════
TRAIN_REAL = os.path.join(EXTRACTED_PATH, "train", "real")
TRAIN_FAKE = os.path.join(EXTRACTED_PATH, "train", "fake")

print("\n[1/4] FaceForensics++")
extract_frames(FF_REAL, TRAIN_REAL)
for p in FF_FAKES:
    extract_frames(p, TRAIN_FAKE)

print("\n[2/4] CelebDF-V2")
for p in CELEB_REALS:
    extract_frames(p, TRAIN_REAL)
extract_frames(CELEB_FAKE, TRAIN_FAKE)

print("\n[3/4] DFD")
extract_frames(DFD_REAL, TRAIN_REAL)
extract_frames(DFD_FAKE, TRAIN_FAKE)

print("\n[4/4] Wild-Deepfake")
extract_frames(WILD_REAL, TRAIN_REAL)
extract_frames(WILD_FAKE, TRAIN_FAKE)

real_cnt = len(os.listdir(TRAIN_REAL))
fake_cnt = len(os.listdir(TRAIN_FAKE))
print(f"\n✅ Extraction complete → real: {real_cnt:,}  fake: {fake_cnt:,}")



[1/4] FaceForensics++


NeuralTextures: 100%|████████████████████████████████| 1000/1000 [00:06<00:00, 146.54it/s]



[2/4] CelebDF-V2


Celeb-synthesis: 100%|██████████████████████████████| 5639/5639 [2:31:27<00:00,  1.61s/it]



[3/4] DFD


DFD_original sequences: 100%|███████████████████████████| 363/363 [25:14<00:00,  4.17s/it]

  ⚠️  No videos in: Datasets\DFD\DFD_manipulated sequences

[4/4] Wild-Deepfake
  ⚠️  No videos in: Datasets\wild-deepfake\real
  ⚠️  No videos in: Datasets\wild-deepfake\fake

✅ Extraction complete → real: 45,041  fake: 132,780


In [2]:
import torch
print(torch.cuda.is_available())      # must print True
print(torch.cuda.get_device_name(0))  # must print your GPU name
print(torch.__version__)              # should show +cu126



True
NVIDIA GeForce RTX 4050 Laptop GPU
2.10.0+cu126


In [14]:
# Optional one-time installs (uncomment if needed)
# !pip install facenet-pytorch timm --quiet

import os, io, cv2, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, WeightedRandomSampler, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from PIL import Image
from sklearn.metrics import f1_score, confusion_matrix, classification_report


In [15]:
# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION  —  adjust paths and hyperparameters here
# ═══════════════════════════════════════════════════════════════

# ── Dataset root (folder containing celeb, DFD, faceforensic, wild-deepfake)
BASE = "Datasets"

# ── Per-dataset sub-paths
FF_REAL       = os.path.join(BASE, "faceforensic", "original")
FF_FAKES      = [os.path.join(BASE, "faceforensic", d)
                 for d in ["Deepfakes","Face2Face","FaceShifter","FaceSwap","NeuralTextures"]]
CELEB_REALS   = [os.path.join(BASE, "celeb", "Celeb-real"),
                 os.path.join(BASE, "celeb", "YouTube-real")]
CELEB_FAKE    = os.path.join(BASE, "celeb", "Celeb-synthesis")
CELEB_TLIST   = os.path.join(BASE, "celeb", "List_of_testing_videos.txt")
DFD_REAL      = os.path.join(BASE, "DFD", "DFD_original sequences")
DFD_FAKE      = os.path.join(BASE, "DFD", "DFD_manipulated sequences")
WILD_REAL     = os.path.join(BASE, "wild-deepfake", "real")
WILD_FAKE     = os.path.join(BASE, "wild-deepfake", "fake")

# ── Frame extraction
EXTRACTED_PATH  = "processed_frames"
IMG_SIZE        = 224       # 380 = EfficientNet-B4 native (needs ~2× VRAM)
FRAME_SKIP      = 5         # Extract every 5th frame
MAX_FRAMES      = 20        # Max frames per video
USE_FACE_DETECT = False     # Set True if facenet-pytorch is installed

# ── Training
BATCH_SIZE      = 64       # Raise to 48-64 with AMP on ≥ 10 GB VRAM
EPOCHS          = 40
PATIENCE        = 5         # Early-stop patience on Val F1
VAL_SPLIT       = 0.15      # 15 % held-out validation set
NUM_WORKERS     = 6
LABEL_SMOOTHING = 0.05
BEST_MODEL_PATH = "best_model.pth"

# ── Inference
INFER_THRESHOLD = 0.5
INFER_FRAMES    = 15

# ── ImageNet stats (correct for pretrained EfficientNet)
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── GPU
cudnn.benchmark     = True
cudnn.deterministic = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device  : cuda
GPU     : NVIDIA GeForce RTX 4050 Laptop GPU
VRAM    : 6.4 GB


In [16]:
USE_FACE_DETECT = True   # was False

In [17]:
# ── Load MTCNN face detector (optional but improves accuracy) ──────────────
face_detector = None
if USE_FACE_DETECT:
    try:
        from facenet_pytorch import MTCNN
        face_detector = MTCNN(
            image_size=IMG_SIZE, margin=30,
            keep_all=False, device=device, post_process=False
        )
        print("✅ MTCNN face detector loaded")
    except ImportError:
        print("⚠️  facenet-pytorch not found → full-frame mode (pip install facenet-pytorch)")
        USE_FACE_DETECT = False
else:
    print("ℹ️  Face detection disabled. Set USE_FACE_DETECT=True to enable.")


✅ MTCNN face detector loaded


In [18]:
def extract_frames(src_folder, dst_folder,
                   frame_skip=FRAME_SKIP, max_frames=MAX_FRAMES):
    """
    Walk src_folder recursively, extract up to max_frames every frame_skip frames.
    Optionally crops to face region via MTCNN.
    Skips videos already extracted (safe to re-run / resume).
    """
    os.makedirs(dst_folder, exist_ok=True)
    VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv", ".flv", ".wmv")

    video_files = [
        os.path.join(r, f)
        for r, _, files in os.walk(src_folder)
        for f in files if f.lower().endswith(VIDEO_EXT)
    ]
    if not video_files:
        print(f"  ⚠️  No videos in: {src_folder}")
        return

    for vpath in tqdm(video_files, desc=os.path.basename(src_folder), ncols=90):
        stem = os.path.splitext(os.path.basename(vpath))[0]
        # Skip if already done
        if sum(1 for f in os.listdir(dst_folder) if f.startswith(stem + "_")) >= max_frames:
            continue

        cap = cv2.VideoCapture(vpath)
        frame_idx, saved = 0, 0

        while cap.isOpened() and saved < max_frames:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_idx % frame_skip == 0:
                if USE_FACE_DETECT and face_detector is not None:
                    try:
                        pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                        ft  = face_detector(pil)         # (C,H,W) uint8 or None
                        if ft is not None:
                            face_np = ft.permute(1,2,0).numpy()
                            frame   = cv2.cvtColor(
                                np.clip(face_np,0,255).astype(np.uint8),
                                cv2.COLOR_RGB2BGR)
                    except Exception:
                        pass  # fall back to full frame

                out = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
                cv2.imwrite(
                    os.path.join(dst_folder, f"{stem}_{saved:04d}.jpg"),
                    out, [cv2.IMWRITE_JPEG_QUALITY, 95]
                )
                saved += 1
            frame_idx += 1

        cap.release()


In [19]:
# ── JPEG noise augmentation (simulates codec artifacts in deepfake videos) ──
class JPEGNoise:
    def __init__(self, quality_range=(60, 95), p=0.4):
        self.quality_range = quality_range
        self.p = p

    def __call__(self, img: Image.Image) -> Image.Image:
        if random.random() < self.p:
            quality = random.randint(*self.quality_range)
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=quality)
            buf.seek(0)
            return Image.open(buf).copy()
        return img


transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    JPEGNoise(quality_range=(60, 95), p=0.4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.10), value="random"),
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

print("✅ Augmentation pipelines ready")


✅ Augmentation pipelines ready


In [20]:
# Load dataset twice with different transforms so val never gets augmented
train_data_src = ImageFolder(os.path.join(EXTRACTED_PATH, "train"), transform=transform_train)
val_data_src   = ImageFolder(os.path.join(EXTRACTED_PATH, "train"), transform=transform_val)

assert train_data_src.class_to_idx == val_data_src.class_to_idx, "Class mismatch!"

total_size = len(train_data_src)
val_size   = int(total_size * VAL_SPLIT)
train_size = total_size - val_size

# Reproducible shuffle-split
indices = list(range(total_size))
random.seed(42)
random.shuffle(indices)
train_indices = indices[val_size:]
val_indices   = indices[:val_size]

train_dataset = Subset(train_data_src, train_indices)
val_dataset   = Subset(val_data_src,   val_indices)

# Class-balanced sampler for training
train_targets  = [train_data_src.targets[i] for i in train_indices]
class_counts   = np.bincount(train_targets)
class_weights  = 1.0 / class_counts
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=4, pin_memory=True, persistent_workers=True,
    prefetch_factor=2   # pre-load 2 batches per worker
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=4, pin_memory=True, persistent_workers=True
)

print(f"Total   : {total_size:,} samples")
print(f"Train   : {train_size:,}  |  Val: {val_size:,}")
print(f"Class   : real={class_counts[0]:,}  fake={class_counts[1]:,}")
print(f"Batches : train={len(train_loader)}  val={len(val_loader)}")


Total   : 177,821 samples
Train   : 151,148  |  Val: 26,673
Class   : real=112,694  fake=38,454
Batches : train=2362  val=209


In [21]:
def build_model(device):
    """
    EfficientNet-B4 pretrained on ImageNet.
    - Freeze first 30 % of params (low-level texture features)
    - Train remaining 70 % + custom head
    - Head: Dropout → Linear(1792→512) → SiLU → BN → Dropout → Linear(512→1)
    """
    m = models.efficientnet_b4(weights="IMAGENET1K_V1")

    # Progressive unfreeze: keep early layers frozen
    params = list(m.parameters())
    freeze_until = int(len(params) * 0.30)
    for i, p in enumerate(params):
        p.requires_grad = (i >= freeze_until)

    # Improved classification head
    in_feat = m.classifier[1].in_features          # 1792 for B4
    m.classifier = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_feat, 512),
        nn.SiLU(),
        nn.BatchNorm1d(512),
        nn.Dropout(p=0.3),
        nn.Linear(512, 1),
    )
    return m.to(device)


model = build_model(device)

# Optional: compile for 10-30 % speedup on PyTorch ≥ 2.0
model = torch.compile(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"Model   : EfficientNet-B4")
print(f"Params  : {trainable:,} trainable / {total_p:,} total ({100*trainable/total_p:.1f} %)")


Model   : EfficientNet-B4
Params  : 18,198,919 trainable / 18,468,169 total (98.5 %)


In [24]:
# ── Label-smoothing BCE loss ────────────────────────────────────────────────
class LabelSmoothingBCE(nn.Module):
    """BCE with label smoothing + class-balanced pos_weight."""
    def __init__(self, smoothing=0.05, pos_weight=None):
        super().__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    def forward(self, logits, targets):
        targets_smooth = targets * (1 - self.smoothing) + 0.5 * self.smoothing
        return self.bce(logits, targets_smooth)


pos_weight = torch.tensor([class_counts[0] / class_counts[1]]).to(device)
criterion  = LabelSmoothingBCE(smoothing=LABEL_SMOOTHING, pos_weight=pos_weight)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=1e-4, betas=(0.9, 0.999)
)

# Cosine annealing with warm restarts — LR naturally recovers after each T_0 cycle
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=1, eta_min=1e-6
)

scaler = GradScaler('cuda')   # AMP gradient scaler

# ── Load best_model.pth and resume training if it exists ────────────────────
start_epoch   = 0
best_val_f1   = 0.0
best_val_loss = float("inf")
early_counter = 0
temporary_models: list = []
if os.path.exists(BEST_MODEL_PATH):
    print(f"🔄 Found {BEST_MODEL_PATH} — attempting to load...")
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device)

    try:
        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            state_dict = ckpt["model_state_dict"]
        else:
            state_dict = ckpt

        # ── Auto-fix key mismatch between compiled / non-compiled checkpoints ──
        model_keys    = set(model.state_dict().keys())
        ckpt_keys     = set(state_dict.keys())
        has_orig_mod  = any(k.startswith("_orig_mod.") for k in model_keys)
        ckpt_has_orig = any(k.startswith("_orig_mod.") for k in ckpt_keys)

        if has_orig_mod and not ckpt_has_orig:
            # Model is compiled, checkpoint is not → add prefix
            state_dict = {"_orig_mod." + k: v for k, v in state_dict.items()}
        elif not has_orig_mod and ckpt_has_orig:
            # Model is not compiled, checkpoint is → strip prefix
            state_dict = {k.replace("_orig_mod.", "", 1): v for k, v in state_dict.items()}

        model.load_state_dict(state_dict)

        if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            scheduler.load_state_dict(ckpt["scheduler_state_dict"])
            scaler.load_state_dict(ckpt["scaler_state_dict"])
            start_epoch   = ckpt.get("epoch", 0)
            best_val_f1   = ckpt.get("best_val_f1", 0.0)
            best_val_loss = ckpt.get("best_val_loss", float("inf"))
            print(f"   ✅ Resumed from epoch {start_epoch} | Best Val F1: {best_val_f1:.4f}")
        else:
            print("   ✅ Loaded legacy weights.")

    except RuntimeError as e:
        print(f"   ⚠️  Checkpoint incompatible: {str(e)[:120]}")
        print(f"   🔁 Starting fresh.")
        backup = BEST_MODEL_PATH.replace(".pth", "_old.pth")
        if os.path.exists(backup):
            os.remove(backup)           # ← remove stale backup first
        os.rename(BEST_MODEL_PATH, backup)
        print(f"   💾 Old checkpoint backed up to: {backup}")
else:
    print("⚡ No checkpoint found — training from scratch.")

🔄 Found best_model.pth — attempting to load...
   ✅ Resumed from epoch 2 | Best Val F1: 0.9812


In [25]:
# Kill current training first with the Stop button, then run this
import torch
import multiprocessing
multiprocessing.set_start_method('spawn', force=True)

In [ ]:
print(f"\n{'='*70}")
print(f"  DEEPFAKE DETECTION TRAINING  |  EfficientNet-B4  |  {device}")
print(f"{'='*70}")
print(f"  Epochs {start_epoch}→{EPOCHS}  |  Batch {BATCH_SIZE}  |  AMP ON  |  Patience {PATIENCE}")
print(f"{'='*70}\n")

for epoch in range(start_epoch, EPOCHS):

    # ── TRAIN ──────────────────────────────────────────────────────────────
    model.train()
    t_loss, t_preds, t_labels = 0.0, [], []

    bar = tqdm(train_loader, desc=f"Ep {epoch+1:02d}/{EPOCHS} train", ncols=90, leave=False)
    for imgs, lbls in bar:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.float().unsqueeze(1).to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast('cuda'):
            out  = model(imgs)
            loss = criterion(out, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        t_loss += loss.item()
        probs  = torch.sigmoid(out.detach())
        preds  = (probs > 0.5).float()
        t_preds.extend(preds.cpu().numpy())
        t_labels.extend(lbls.cpu().numpy())
        bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = t_loss / len(train_loader)
    train_f1   = f1_score(t_labels, t_preds, zero_division=0)

    # ── VALIDATE ───────────────────────────────────────────────────────────
    model.eval()
    v_loss, v_preds, v_labels = 0.0, [], []

    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc=f"Ep {epoch+1:02d}/{EPOCHS} val  ", ncols=90, leave=False):
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.float().unsqueeze(1).to(device, non_blocking=True)
            with autocast('cuda'):
                out  = model(imgs)
                loss = criterion(out, lbls)
            v_loss += loss.item()
            probs = torch.sigmoid(out)
            preds = (probs > 0.5).float()
            v_preds.extend(preds.cpu().numpy())
            v_labels.extend(lbls.cpu().numpy())

    val_loss = v_loss / len(val_loader)
    val_f1   = f1_score(v_labels, v_preds, zero_division=0)

    scheduler.step(epoch + 1)
    lr = optimizer.param_groups[0]["lr"]
    flag = " 🔥" if val_f1 > best_val_f1 else ""

    print(
        f"Ep {epoch+1:02d}/{EPOCHS} | "
        f"Train  loss={train_loss:.4f} f1={train_f1:.4f} | "
        f"Val  loss={val_loss:.4f} f1={val_f1:.4f}{flag} | "
        f"lr={lr:.2e}"
    )

    # ── SAVE TEMP CHECKPOINT ───────────────────────────────────────────────
    tmp_path = f"temp_ep{epoch+1:02d}_valf1_{val_f1:.4f}.pth"
    torch.save(model.state_dict(), tmp_path)
    temporary_models.append((tmp_path, val_f1))

    # ── BEST MODEL CHECKPOINT ─────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1   = val_f1
        best_val_loss = val_loss
        early_counter = 0

        torch.save({
            "epoch"               : epoch + 1,
            "model_state_dict"    : model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict"   : scaler.state_dict(),
            "best_val_f1"         : best_val_f1,
            "best_val_loss"       : best_val_loss,
            "train_f1"            : train_f1,
            "train_loss"          : train_loss,
        }, BEST_MODEL_PATH)
        print(f"   💾 Saved best model → {BEST_MODEL_PATH}  (Val F1: {best_val_f1:.4f})")

    else:
        early_counter += 1
        print(f"   ⚠️  No improvement  [{early_counter}/{PATIENCE}]")
        if early_counter >= PATIENCE:
            print(f"\n🛑 Early stopping at epoch {epoch+1}.")
            break

print(f"\n{'='*70}")
print(f"  Training done!  Best Val F1: {best_val_f1:.4f}")
print(f"{'='*70}")



  DEEPFAKE DETECTION TRAINING  |  EfficientNet-B4  |  cuda
  Epochs 2→40  |  Batch 64  |  AMP ON  |  Patience 5



Ep 03/40 train:   0%|                                            | 0/2362 [00:00<?, ?it/s]

In [ ]:
# ═══════════════════════════════════════════════════════
#  POST-TRAINING: pick the absolute best temp checkpoint
#  by re-evaluating every saved temp model on val set
# ═══════════════════════════════════════════════════════
print("\n🔍 Re-evaluating all temp checkpoints on validation set...")
print("-" * 60)

final_best_f1   = 0.0
final_best_file = None

for tmp_path, recorded_f1 in temporary_models:
    if not os.path.exists(tmp_path):
        continue

    model.load_state_dict(torch.load(tmp_path, map_location=device))
    model.eval()
    preds_list, labels_list = [], []

    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.float().unsqueeze(1).to(device, non_blocking=True)
            with autocast('cuda'):
                out = model(imgs)
            preds = (torch.sigmoid(out) > 0.5).float()
            preds_list.extend(preds.cpu().numpy())
            labels_list.extend(lbls.cpu().numpy())

    f1  = f1_score(labels_list, preds_list, zero_division=0)
    tag = " ← BEST" if f1 > final_best_f1 else ""
    print(f"  {tmp_path:<50} Val F1: {f1:.4f}{tag}")

    if f1 > final_best_f1:
        final_best_f1   = f1
        final_best_file = tmp_path

# Update best_model.pth if a temp model beat it
if final_best_file and final_best_f1 > best_val_f1:
    model.load_state_dict(torch.load(final_best_file, map_location=device))
    torch.save({
        "epoch"               : EPOCHS,
        "model_state_dict"    : model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict"   : scaler.state_dict(),
        "best_val_f1"         : final_best_f1,
        "best_val_loss"       : best_val_loss,
    }, BEST_MODEL_PATH)
    print(f"\n🏆 Better checkpoint found: {final_best_file} (Val F1: {final_best_f1:.4f})")
    print(f"   💾 Updated {BEST_MODEL_PATH}")
else:
    print(f"\n✅ {BEST_MODEL_PATH} already holds the best checkpoint (Val F1: {best_val_f1:.4f})")


In [ ]:
# ── Remove temporary per-epoch checkpoints, keep only best_model.pth ───────
removed = 0
for tmp_path, _ in temporary_models:
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
        removed += 1

# Also sweep any stray .pth files in cwd (except best_model.pth)
for f in os.listdir("."):
    if f.endswith(".pth") and f != BEST_MODEL_PATH:
        os.remove(f)
        removed += 1

temporary_models.clear()
print(f"🧹 Cleaned up {removed} temp file(s).  Kept: {BEST_MODEL_PATH}")


In [ ]:
def load_best_model(path=BEST_MODEL_PATH):
    """Load best checkpoint for inference. Returns model in eval mode."""
    assert os.path.exists(path), f"Checkpoint not found: {path}"
    m    = build_model(device)
    ckpt = torch.load(path, map_location=device)
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        m.load_state_dict(ckpt["model_state_dict"])

        ep = ckpt.get("epoch", "?")
        f1 = ckpt.get("best_val_f1", 0)
        print(f"✅ Loaded checkpoint  epoch={ep}  Val F1={f1:.4f}")
    else:
        m.load_state_dict(ckpt)
        print("✅ Loaded legacy weight-only checkpoint")
    m.eval()
    return m


# TTA (Test-Time Augmentation) transforms
_tta_transforms = [
    transform_val,
    transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ]),
]


def predict_video(video_path, model=None, n_frames=INFER_FRAMES,
                  frame_skip=10, threshold=INFER_THRESHOLD, tta=True):
    """
    Predict REAL/FAKE for a single video.
    Uses Test-Time Augmentation (TTA) by default.

    Returns:
        prob       (float) : probability of being FAKE
        label      (str)   : 'FAKE' or 'REAL'
        confidence (float) : % confidence in the decision
    """
    if model is None:
        model = load_best_model()
    model.eval()

    tfms  = _tta_transforms if tta else [transform_val]
    cap   = cv2.VideoCapture(video_path)
    probs = []
    count = 0

    while cap.isOpened() and len(probs) < n_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if count % frame_skip == 0:
            pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).resize((IMG_SIZE, IMG_SIZE))
            fp  = []
            for tfm in tfms:
                t = tfm(pil).unsqueeze(0).to(device)
                with torch.no_grad(), autocast('cuda'):
                    fp.append(torch.sigmoid(model(t)).item())
            probs.append(float(np.mean(fp)))
        count += 1

    cap.release()
    if not probs:
        return 0.5, "UNKNOWN", 50.0

    p          = float(np.mean(probs))
    label      = "FAKE" if p > threshold else "REAL"
    confidence = p * 100 if label == "FAKE" else (1 - p) * 100
    return p, label, confidence


def predict_image(image_path, model=None, threshold=INFER_THRESHOLD, tta=True):
    """Predict REAL/FAKE for a single image file."""
    if model is None:
        model = load_best_model()
    model.eval()

    pil  = Image.open(image_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    tfms = _tta_transforms if tta else [transform_val]
    fp   = []
    for tfm in tfms:
        t = tfm(pil).unsqueeze(0).to(device)
        with torch.no_grad(), autocast('cuda'):
            fp.append(torch.sigmoid(model(t)).item())

    p          = float(np.mean(fp))
    label      = "FAKE" if p > threshold else "REAL"
    confidence = p * 100 if label == "FAKE" else (1 - p) * 100
    return p, label, confidence


In [ ]:
# ─── Quick test on a single file ───────────────────────────────────────────
# Uncomment and set path to test

# VIDEO:
# prob, label, conf = predict_video("path/to/test.mp4")
# print(f"Result : {label}  ({conf:.1f} % confident)  raw prob={prob:.4f}")

# IMAGE:
# prob, label, conf = predict_image("path/to/face.jpg")
# print(f"Result : {label}  ({conf:.1f} % confident)  raw prob={prob:.4f}")


In [ ]:
def test_celeb_df(model=None):
    """
    Evaluate on CelebDF-V2.
    Uses official test split from List_of_testing_videos.txt when available.
    """
    if model is None:
        model = load_best_model()

    all_preds, all_labels = [], []

    # Load official test list
    test_set = set()
    if os.path.exists(CELEB_TLIST):
        with open(CELEB_TLIST) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    test_set.add(os.path.basename(parts[1]))
        print(f"Using {len(test_set)} official CelebDF-V2 test videos")

    def want(fname):
        return (not test_set) or (os.path.basename(fname) in test_set)

    VIDEO_EXT = (".mp4", ".avi", ".mov")

    # Real
    for folder in CELEB_REALS:
        if not os.path.exists(folder):
            continue
        files = [f for f in os.listdir(folder) if f.lower().endswith(VIDEO_EXT)]
        for f in tqdm(files, desc=f"Celeb-real {os.path.basename(folder)}"):
            vpath = os.path.join(folder, f)
            if not want(vpath):
                continue
            prob, _, _ = predict_video(vpath, model=model)
            all_preds.append(1 if prob > INFER_THRESHOLD else 0)
            all_labels.append(0)

    # Fake
    if os.path.exists(CELEB_FAKE):
        files = [f for f in os.listdir(CELEB_FAKE) if f.lower().endswith(VIDEO_EXT)]
        for f in tqdm(files, desc="Celeb-synthesis"):
            vpath = os.path.join(CELEB_FAKE, f)
            if not want(vpath):
                continue
            prob, _, _ = predict_video(vpath, model=model)
            all_preds.append(1 if prob > INFER_THRESHOLD else 0)
            all_labels.append(1)

    if not all_preds:
        print("⚠️  No test videos found!")
        return

    print("\n" + "="*55)
    print("  CELEB-DF V2 TEST RESULTS")
    print("="*55)
    print(confusion_matrix(all_labels, all_preds))
    print()
    print(classification_report(all_labels, all_preds, target_names=["Real","Fake"]))

test_celeb_df()


In [ ]:
def test_wild_deepfake(model=None):
    """Evaluate on Wild-Deepfake (contains image sequences and/or short clips)."""
    if model is None:
        model = load_best_model()

    all_preds, all_labels = [], []
    IMG_EXT   = (".jpg", ".jpeg", ".png")
    VIDEO_EXT = (".mp4", ".avi", ".mov")

    for label_val, folder, tag in [(0, WILD_REAL, "real"), (1, WILD_FAKE, "fake")]:
        if not os.path.exists(folder):
            print(f"⚠️  Folder not found: {folder}")
            continue

        files = os.listdir(folder)
        for f in tqdm(files, desc=f"Wild-Deepfake {tag}"):
            fpath = os.path.join(folder, f)
            fl    = f.lower()

            if fl.endswith(VIDEO_EXT):
                prob, _, _ = predict_video(fpath, model=model)
            elif fl.endswith(IMG_EXT):
                prob, _, _ = predict_image(fpath, model=model)
            else:
                continue

            all_preds.append(1 if prob > INFER_THRESHOLD else 0)
            all_labels.append(label_val)

    if not all_preds:
        print("⚠️  No files found for Wild-Deepfake testing!")
        return

    print("\n" + "="*55)
    print("  WILD-DEEPFAKE TEST RESULTS")
    print("="*55)
    print(confusion_matrix(all_labels, all_preds))
    print()
    print(classification_report(all_labels, all_preds, target_names=["Real","Fake"]))

test_wild_deepfake()
